In [ ]:
# =============================================================================
# AI INFRASTRUCTURE — FOUNDATIONAL DATA PULLS & VISUALIZATIONS
# Run locally in Jupyter or as a script.
# Install: pip install pandas matplotlib seaborn requests
# Data: Alpha Vantage (free key at alphavantage.co — 25 calls/day on free tier)
# =============================================================================


# =============================================================================
# CELL 1 — IMPORTS, CONFIG & API KEY
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import requests
import time
import warnings
warnings.filterwarnings("ignore")

# !! REPLACE WITH YOUR NEW KEY !!
AV_KEY = ""
AV_BASE = "https://www.alphavantage.co/query"

# Plot style
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 130, "figure.figsize": (12, 5)})

# Universe — grouped by stack layer
UNIVERSE = {
    "Accelerators":  ["NVDA", "AMD", "AVGO"],
    "Networking":    ["ANET", "CSCO"],
    "Servers":       ["SMCI", "DELL", "HPE"],
    "Hyperscalers":  ["MSFT", "AMZN", "GOOGL", "META"],
    "Power":         ["CEG", "VST", "NRG"],
    "Power Equip":   ["VRT", "ETN"],
    "Colocation":    ["EQIX", "DLR"],
}
ALL_TICKERS = [t for tickers in UNIVERSE.values() for t in tickers]


In [5]:
# =============================================================================
# CELL 2 — ALPHA VANTAGE PRICE PULL (monthly, adjusted close)
# Free tier: 25 calls/day — this pulls all 19 tickers so spread across 2 days
# if on the free tier, or upgrade to premium ($50/mo) for unlimited calls.
# Tip: run once, save to CSV, reload from CSV on subsequent runs.
# =============================================================================

def fetch_monthly_av(ticker, api_key, retries=3):
    """Pull monthly adjusted close from Alpha Vantage. Returns a Series."""
    params = {
        "function": "TIME_SERIES_MONTHLY_ADJUSTED",
        "symbol": ticker,
        "apikey": api_key,
        "outputsize": "full",
    }
    for attempt in range(retries):
        try:
            r = requests.get(AV_BASE, params=params, timeout=15)
            data = r.json()
            if "Monthly Adjusted Time Series" not in data:
                print(f"  {ticker}: unexpected response — {list(data.keys())}")
                return None
            series = data["Monthly Adjusted Time Series"]
            s = pd.Series(
                {pd.to_datetime(k): float(v["5. adjusted close"])
                 for k, v in series.items()}
            ).sort_index()
            return s
        except Exception as e:
            print(f"  {ticker} attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return None

def load_prices(tickers, api_key, cache_file="price_cache.csv"):
    """
    Load from cache CSV if it exists, otherwise pull from Alpha Vantage.
    Re-pull only when you want fresh data by deleting price_cache.csv.
    """
    import os
    if os.path.exists(cache_file):
        print(f"Loading from cache: {cache_file}")
        return pd.read_csv(cache_file, index_col=0, parse_dates=True)

    print("Pulling from Alpha Vantage (this may take a few minutes)...")
    frames = {}
    for i, ticker in enumerate(tickers):
        print(f"  [{i+1}/{len(tickers)}] {ticker}")
        s = fetch_monthly_av(ticker, api_key)
        if s is not None:
            frames[ticker] = s
        # AV free tier: max 5 calls/min — sleep between calls
        time.sleep(13)

    df = pd.DataFrame(frames).sort_index()
    df.to_csv(cache_file)
    print(f"Saved to {cache_file}")
    return df

# --- Pull or load prices ---
prices = load_prices(ALL_TICKERS, AV_KEY)

# Filter to last 2 years
prices = prices[prices.index >= prices.index.max() - pd.DateOffset(years=2)]
print(f"\nLoaded {len(prices.columns)} tickers, {len(prices)} months")
print(prices.tail(3))

Pulling from Alpha Vantage (this may take a few minutes)...
  [1/19] NVDA
  [2/19] AMD
  [3/19] AVGO
  [4/19] ANET
  [5/19] CSCO
  [6/19] SMCI
  [7/19] DELL
  [8/19] HPE
  [9/19] MSFT
  [10/19] AMZN
  [11/19] GOOGL
  [12/19] META
  [13/19] CEG
  [14/19] VST
  [15/19] NRG
  [16/19] VRT
  [17/19] ETN
  [18/19] EQIX
  [19/19] DLR
Saved to price_cache.csv

Loaded 19 tickers, 25 months
                NVDA     AMD      AVGO    ANET      CSCO   SMCI    DELL  \
2026-05-29  210.8989  516.10  446.0307  159.47  119.9779  46.09  420.91   
2026-06-30  200.0900  580.91  377.7500  169.88  117.0288  29.33  431.46   
2026-07-15  212.5000  529.14  394.2800  171.92  111.7700  26.89  412.68   

                HPE    MSFT    AMZN     GOOGL     META      CEG       VST  \
2026-05-29  42.9136  450.24  270.64  380.1098  631.951  287.750  160.0109   
2026-06-30  45.1100  373.02  238.34  357.3700  563.290  248.370  158.6300   
2026-07-15  47.3900  395.63  254.96  370.9200  681.310  258.115  160.2300   

      